# 12 – SaaS Billing Patterns: Fixed & Dynamic Costs

Every AI SaaS platform has two cost shapes:

- **Dynamic inference** — you don't know the cost until the model returns. A chat call might use 200 tokens or 2000. You must *reserve* a worst-case hold before the call, then *settle* the actual cost when it completes.
- **Fixed-cost batch jobs** — the cost is known in advance (generating a PDF report always costs 10 credits). A single atomic deduction is the right pattern.

This notebook walks both patterns end-to-end, then layers in the operational concerns every SaaS needs: free-tier allowances, double-submit prevention, feature gating, advisory balance reads, paid-user overdraft, and refunds. The code cells are production-shaped — copy the patterns you need.

**What we cover:**
1. Pricing setup — models + fixed jobs + plan tiers
2. Dynamic inference with monthly allowance (free tier)
3. Allowance exhaustion → balance fallback
4. Fixed-cost batch job with idempotency
5. Concurrency control — preventing double-submit
6. Feature gating — plan-tier enforcement
7. Advisory reads — balance bar and send-button logic
8. Paid users with overdraft
9. `run_billed` — the one-call shortcut
10. Refunds

In [ ]:
import uuid
from decimal import Decimal

from ducto import (
    CreditManager,
    UsageMetrics,
    ConcurrencyLimitError,
    FeatureNotEntitledError,
    InsufficientCreditsError,
)
from ducto.interface.models import PricingConfigData, PlanDefinition, OperationPolicy
from shared import start_postgres_store, cleanup

store, pgdata = start_postgres_store()
manager = CreditManager(store=store)
print("✔ PostgresStore ready.")

## Pricing setup

A real platform has multiple model tiers and some fixed-cost jobs alongside token-billed inferences. We define three plan tiers:

- **Free** — 100 credits/month allowance, 1 concurrent chat, no agentic.
- **Pro** — 500 credits/month, 4 concurrent, agentic unlocked.
- **Paid** — no allowance, overdraft down to -50, 8 concurrent, all features.

The `_default` model expression (`input_tokens * 1`) gives 1 credit per token — easy mental arithmetic for the demo. The `gpt-4o` expression uses more realistic per-token pricing.

In [ ]:
manager.publish_pricing_from_dict({
    "version": 1,
    "models": {
        "gpt-4o":    "input_tokens * 0.01 + output_tokens * 0.03",
        "_default":  "input_tokens * 1 + output_tokens * 1",
    },
    "fixed": {
        "daily_report": 10,
        "batch_train":  50,
    },
    "min_balance": 0,
    "plans": {
        "free": {
            "id": "free",
            "name": "Free",
            "free_allowance": 100,
            "max_concurrent": 1,
            "features": {"chat": True},
        },
        "pro": {
            "id": "pro",
            "name": "Pro",
            "free_allowance": 500,
            "max_concurrent": 4,
            "features": {"chat": True, "agentic": True},
        },
        "paid": {
            "id": "paid",
            "name": "Paid",
            "free_allowance": 0,
            "max_concurrent": 8,
            "features": {"chat": True, "agentic": True},
        },
    },
})
print("✔ Pricing published.")

## 1. Dynamic inference — free tier with monthly allowance

Chat is the classic dynamic-cost operation: the token count isn't known until the model returns. The safe pattern is:

1. **`reserve`** a worst-case hold before sending to the model.
2. Call the model.
3. **`settle`** the actual token count. The unused portion of the hold is freed automatically.

Free-tier users get 100 credits of monthly allowance. `deduct_with_allowance` consumes allowance first — the balance is only touched once the allowance runs out. `DeductionResult.allowance_consumed` tells you how much of this charge was covered by allowance.

The `_default` model expression is `input_tokens * 1 + output_tokens * 1`, so 1 credit per token on either side. The worst-case reserve below is 1000 + 800 = **1800 credits**. The `reserve` call succeeds because `create_lease` is allowance-aware: it adds the 100-credit free allowance to the effective headroom (balance 200 + allowance 100 = 300 ≥ 1800? — no, the user also has 200 balance so effective = 200 + 100 = 300 < 1800). The example below sizes the hold to fit within the available headroom to demonstrate the full flow; for a real chat integration you would top up the balance or size the worst-case to your expected max context.

In [ ]:
free_user = str(uuid.uuid4())
manager.add_credits(free_user, Decimal("200"), "signup_bonus")
manager.set_user_plan(free_user, "free")

# Check remaining allowance before presenting the send button.
allowance = manager.check_allowance(free_user)  # via manager, not store directly
print(f"Allowance remaining: {allowance.allowance_remaining} / period ends {allowance.period_end[:10]}")

# Step 1 — reserve worst-case (100 input + 100 output on _default = 200 credits).
# Effective headroom = balance(200) + allowance(100) = 300, so 200 fits.
worst = UsageMetrics(model="_default", input_tokens=100, output_tokens=100)
lease = manager.reserve(free_user, worst, operation_type="chat")
print(f"\nLease held:      {lease.amount} credits")
print(f"Available after: {lease.available} credits")

# Step 2 — call the model (simulated). Actual: 60 input + 40 output = 100 credits.
actual = UsageMetrics(model="_default", input_tokens=60, output_tokens=40)

# Step 3 — settle the actual cost.
ded = manager.settle(free_user, lease.lease_id, actual)
print(f"\nActual charged:    {ded.amount}")
print(f"From allowance:    {ded.allowance_consumed}")
print(f"From balance:      {ded.amount - ded.allowance_consumed}")
print(f"Balance after:     {ded.balance_after}")

allowance = manager.check_allowance(free_user)
print(f"Allowance remaining: {allowance.allowance_remaining}")

## 2. Allowance exhaustion — balance fallback

Once the monthly allowance is used up, charges fall through to the balance. The transition is seamless — the same `reserve` → `settle` flow works regardless. `allowance_consumed` shows how much was covered by allowance in each call; the remainder comes from the balance.

In [ ]:
# After the previous 100-credit call: allowance_consumed = 100 (all of it),
# balance was untouched. Allowance is now zero.
# This call shows pure-balance billing.

allowance = manager.check_allowance(free_user)
print(f"Allowance left before this call: {allowance.allowance_remaining}")

# Second chat — no allowance left, entirely from balance.
# 50 input + 50 output = 100 credits; balance = 200, so it fits.
lease2 = manager.reserve(free_user, UsageMetrics(model="_default", input_tokens=50, output_tokens=50), operation_type="chat")
ded2 = manager.settle(free_user, lease2.lease_id, UsageMetrics(model="_default", input_tokens=50, output_tokens=50))

print(f"Charged:        {ded2.amount}")
print(f"From allowance: {ded2.allowance_consumed}  ← 0 now, allowance exhausted")
print(f"From balance:   {ded2.amount - ded2.allowance_consumed}")
print(f"Balance after:  {ded2.balance_after}")

## 3. Fixed-cost batch job

Operations with a known price — generating a PDF report, running a training job, sending a bulk notification batch — use `deduct_fixed`. The job name maps to the `fixed` section of your pricing config. There is no estimate/settle cycle: the cost is deducted atomically.

**Critical ordering rule: deduct first, execute second.**
Unlike dynamic inference (where you must wait for the model to know the cost), a fixed-cost job's price is known up front — so the debit happens *before* the job starts. If you flip the order (run job → charge), a user can exhaust their balance mid-run and the completed work goes unpaid.

**Job failure → refund.** Because the charge is taken before execution, any failure in the job must be followed by `refund_credits` using the `transaction_id` from the `DeductionResult`. The example below shows the full success path and the failure path with refund.

`deduct_fixed` is idempotent when you pass an `idempotency_key` — retrying the same key returns the original result instead of double-charging.

In [ ]:
batch_user = str(uuid.uuid4())
manager.add_credits(batch_user, Decimal("500"), "purchase")

# ── Success path ──────────────────────────────────────────────────────────────
# Step 1: deduct BEFORE starting the job.
idem_key = f"report-{batch_user}-2026-06-30"
result = manager.deduct_fixed(batch_user, "daily_report", idempotency_key=idem_key)
print(f"Report charged: {result.amount} credits  (balance: {result.balance_after})")

# Step 2: run the job only after the deduction succeeds.
def generate_report():
    return {"pages": 3, "status": "ok"}  # simulated

report = generate_report()
print(f"Report generated: {report}")

# Retry with same key — idempotent, no double charge.
result2 = manager.deduct_fixed(batch_user, "daily_report", idempotency_key=idem_key)
print(f"Retry (idempotent): amount={result2.amount}, idempotent={result2.idempotent}, balance={result2.balance_after}")

# ── Failure path ──────────────────────────────────────────────────────────────
# Step 1: deduct before kicking off the training job.
train = manager.deduct_fixed(batch_user, "batch_train")
print(f"\nTraining job charged: {train.amount} credits  (balance: {train.balance_after})")

# Step 2: job execution fails partway through.
def run_training():
    raise RuntimeError("GPU node unreachable")

try:
    run_training()
except RuntimeError as e:
    print(f"Job failed: {e}")
    # Step 3: refund the deduction — credits are restored to the user.
    refund = manager.refund_credits(train.transaction_id, reason="training_job_failed")
    assert refund.error is None, f"Refund failed: {refund.error}"
    print(f"Refund issued: +{refund.amount} credits  (balance: {refund.new_balance})")

## 4. Concurrency control — preventing double-submit

An impatient user who clicks *Send* twice, or a client that retries before the first response arrives, creates two simultaneous leases for the same operation. `max_concurrent` caps the number of active leases per operation type per user. The second admission is rejected with `ConcurrencyLimitError` — before any model call is made.

The free plan already has `max_concurrent: 1` for chat. The slot is freed when the first lease is settled or released.

In [ ]:
conc_user = str(uuid.uuid4())
manager.add_credits(conc_user, Decimal("200"), "signup_bonus")
manager.set_user_plan(conc_user, "free")  # max_concurrent=1 for chat

# First request lands — lease acquired.
first = manager.reserve(conc_user, UsageMetrics(model="_default", input_tokens=100), operation_type="chat")
print(f"First request: lease acquired ({first.lease_id[:8]}…)")

# Second request (double-submit) — rejected immediately.
try:
    manager.reserve(conc_user, UsageMetrics(model="_default", input_tokens=100), operation_type="chat")
except ConcurrencyLimitError as e:
    print(f"Second request: ConcurrencyLimitError — {e}")

# First request finishes — slot freed, next request can proceed.
manager.settle(conc_user, first.lease_id, UsageMetrics(model="_default", input_tokens=80))
third = manager.reserve(conc_user, UsageMetrics(model="_default", input_tokens=100), operation_type="chat")
print(f"After settle:  new lease acquired ({third.lease_id[:8]}…)")
manager.release(conc_user, third.lease_id)

## 5. Feature gating — plan-tier enforcement

Expensive or risky operations (autonomous agentic runs, bulk exports, higher-context models) should be restricted to plans that include them. Pass `required_feature` to `reserve` — ducto checks the user's plan and raises `FeatureNotEntitledError` before any work starts or credits are held.

In [ ]:
free_gated = str(uuid.uuid4())
pro_user   = str(uuid.uuid4())
manager.add_credits(free_gated, Decimal("500"), "signup_bonus")
manager.add_credits(pro_user,   Decimal("500"), "purchase")
manager.set_user_plan(free_gated, "free")
manager.set_user_plan(pro_user,   "pro")

# Free user — agentic feature absent from plan.
try:
    manager.reserve(
        free_gated,
        UsageMetrics(model="_default", input_tokens=500),
        operation_type="agentic",
        required_feature="agentic",
    )
except FeatureNotEntitledError:
    print("Free user: FeatureNotEntitledError — agentic not in plan (no hold placed)")

# Pro user — feature present, admission succeeds.
pro_lease = manager.reserve(
    pro_user,
    UsageMetrics(model="_default", input_tokens=500),
    operation_type="agentic",
    required_feature="agentic",
)
print(f"Pro user:   lease acquired — {pro_lease.amount} credits held")
manager.release(pro_user, pro_lease.lease_id)

## 6. Advisory reads — balance bar and send-button logic

For UI elements that show the user their remaining balance or decide whether to enable the *Send* button, use the non-locking advisory reads. They are fast, never block the operation, and may be slightly stale under concurrency — that's fine for UI. Only `reserve` is the real admission gate.

- `get_available()` → `balance`, `reserved` (active lease holds), `available = balance − reserved`
- `can_afford(amount_or_metrics)` → `affordable`, `available`, `worst_case`, `reason`

In [ ]:
ui_user = str(uuid.uuid4())
manager.add_credits(ui_user, Decimal("150"), "purchase")

# Simulate one in-flight lease already holding 80 credits.
in_flight = manager.reserve(ui_user, Decimal("80"), operation_type="chat")

# Balance bar: show how much is available for new work.
avail = manager.get_available(ui_user)
print("Balance bar:")
print(f"  Total balance:   {avail.balance}")
print(f"  In-flight holds: {avail.reserved}")
print(f"  Available now:   {avail.available}")

# Send-button affordability check.
next_request = UsageMetrics(model="_default", input_tokens=200, output_tokens=100)  # worst-case 300
check = manager.can_afford(ui_user, next_request)
print(f"\nSend button enabled: {check.affordable}")
print(f"  Worst-case cost: {check.worst_case}")
print(f"  Available:       {check.available}")

# Request that would exceed available funds.
too_big = manager.can_afford(ui_user, Decimal("200"))
print(f"\n200-credit request affordable: {too_big.affordable}  (reason: {too_big.reason})")

manager.release(ui_user, in_flight.lease_id)

## 7. Paid users with overdraft

Trusted paid users with auto-reload cards can run past their balance — useful when the model returns more tokens than estimated and you never want to truncate a live response. The `overdraft` preset admits down to a negative floor; `settle` bills the full actual cost even if it exceeds the hold. New admissions are blocked once the floor is hit, until `add_credits` reconciles the balance.

The `paid` plan has `default_billing_mode: overdraft` and `overdraft_floor: -50`. Any user on the paid plan automatically gets overdraft treatment.

In [ ]:
# A dedicated manager for overdraft users — floor matches the plan.
od_mgr = CreditManager(store=store, policy="overdraft", overdraft_floor=Decimal("-50"))
od_mgr.publish_pricing_from_dict({
    "version": 1,
    "models": {"_default": "input_tokens * 1 + output_tokens * 1"},
    "min_balance": 0,
})

paid_user = str(uuid.uuid4())
od_mgr.add_credits(paid_user, Decimal("30"), "purchase")  # thin balance

# Reserve an estimate — work may cost more.
lease = od_mgr.reserve(paid_user, Decimal("20"), operation_type="chat")
print(f"Estimated hold: {lease.amount}  balance available: {lease.available}")

# Model returns 60 tokens — actual > estimate; settle bills the full 60 (de-clamped).
ded = od_mgr.settle(paid_user, lease.lease_id, Decimal("60"))
print(f"Actual charged: {ded.amount}  balance after: {ded.balance_after}")

# Balance is now negative. New admission blocked until reconciled.
try:
    od_mgr.reserve(paid_user, Decimal("5"), operation_type="chat")
except InsufficientCreditsError:
    print("New request blocked — InsufficientCreditsError (balance below floor)")

# Auto-reload fires (card charged), balance reconciled.
od_mgr.add_credits(paid_user, Decimal("200"), "auto_reload")
after = od_mgr.get_available(paid_user)
print(f"After reload:   balance={after.balance}  available={after.available}")

# Now admission succeeds again.
new_lease = od_mgr.reserve(paid_user, Decimal("10"), operation_type="chat")
od_mgr.release(paid_user, new_lease.lease_id)
print("New request admitted successfully.")

## 8. `run_billed` — the one-call shortcut

Wiring `reserve → work → settle/release` by hand is repetitive and easy to get wrong (especially forgetting the `release` on exception). `run_billed` does it for you: pass an `estimate` and a zero-arg `do_work` callable that returns `(result, actual_metrics)`. On any exception the lease is auto-released and the error re-raised — no orphaned holds.

Use `run_billed` for straightforward inference calls. Use the explicit three-step form when you need mid-flight `renew`, per-step metrics accumulation, or custom error handling.

In [ ]:
rb_user = str(uuid.uuid4())
manager.add_credits(rb_user, Decimal("500"), "purchase")
manager.set_user_plan(rb_user, "pro")

def call_model():
    # Your actual model call goes here.
    response = "The capital of France is Paris."
    actual_usage = UsageMetrics(model="_default", input_tokens=40, output_tokens=12)
    return response, actual_usage

out = manager.run_billed(
    rb_user,
    estimate=UsageMetrics(model="_default", input_tokens=200, output_tokens=100),
    do_work=call_model,
    operation_type="chat",
)
print(f"Result:         {out['result']!r}")
print(f"Actual cost:    {out['deduction'].amount}")
print(f"Balance after:  {out['deduction'].balance_after}")

# Auto-release on failure — no leaked hold.
def failing_model():
    raise RuntimeError("model timeout")

try:
    manager.run_billed(rb_user, estimate=Decimal("50"), do_work=failing_model)
except RuntimeError as e:
    avail = manager.get_available(rb_user)
    print(f"\nAfter failure:  error='{e}'  reserved={avail.reserved}  (lease auto-released)")

## 9. Refunds

Refunds reverse a completed deduction and restore the user's balance. They are identified by `transaction_id` from the `DeductionResult`. Partial refunds are supported. `result.error` is set (not raised) on business-rule failures such as over-refunding, duplicate refunds, or refunding a purchase — always check it before treating the result as success.

In [ ]:
ref_user = str(uuid.uuid4())
manager.add_credits(ref_user, Decimal("500"), "purchase")

# Charge for a batch training job.
ded = manager.deduct_fixed(ref_user, "batch_train")
print(f"Charged:        {ded.amount}  (tx: {ded.transaction_id[:8]}…  balance: {ded.balance_after})")

# Full refund — training job failed before completing.
refund = manager.refund_credits(ded.transaction_id, reason="training_job_failed")
assert refund.error is None, f"Refund failed: {refund.error}"
print(f"Full refund:    +{refund.amount}  new balance: {refund.new_balance}")

# Partial refund example — dynamic inference, bill only for tokens actually processed.
lease = manager.reserve(ref_user, Decimal("100"), operation_type="chat")
ded2 = manager.settle(ref_user, lease.lease_id, Decimal("80"))
print(f"\nCharged (inference): {ded2.amount}  balance: {ded2.balance_after}")

partial = manager.refund_credits(ded2.transaction_id, amount=Decimal("30"), reason="user_cancelled_mid_stream")
assert partial.error is None
print(f"Partial refund:      +{partial.amount}  new balance: {partial.new_balance}")

# Duplicate refund — business-rule failure, never raises.
dup = manager.refund_credits(ded.transaction_id)  # already fully refunded
print(f"\nDuplicate refund:    error='{dup.error}'  (no credits moved)")

## Recap

| Use case | Pattern | Key method |
|---|---|---|
| Dynamic inference (unknown token count) | `reserve` → model call → `settle` (or `release` on error) | `manager.reserve()` / `manager.settle()` |
| One-call dynamic inference | `run_billed` auto-wires reserve/settle/release | `manager.run_billed()` |
| Fixed-cost job (known price) | Single atomic deduct, idempotent, **does not consume inference allowance** | `manager.deduct_fixed()` |
| Free monthly credits | Plan with `free_allowance`; check via manager | `manager.check_allowance()` |
| Double-submit prevention | Plan or manager `max_concurrent` per op type | Raises `ConcurrencyLimitError` |
| Plan-gated features | Pass `required_feature` to `reserve` | Raises `FeatureNotEntitledError` |
| Balance bar / send button | Advisory reads include allowance headroom | `manager.get_available()` / `manager.can_afford()` |
| Paid users, auto-reload | Overdraft preset + `on_low_balance` hook | `policy="overdraft"` + `manager.add_credits()` |
| Reverse a charge | Refund by transaction ID, supports partial | `manager.refund_credits()` |

**Error reference:**
- `InsufficientCreditsError` — balance + allowance (minus active holds) below floor at admission
- `ConcurrencyLimitError` — `max_concurrent` active leases for this op type already
- `FeatureNotEntitledError` — user's plan missing `required_feature`
- `CapReachedError` — spend cap (deny) hit at admission
- `LeaseExpiredError` — lease TTL elapsed before `settle`
- `LeaseNotFoundError` — `lease_id` unknown or already released

**Event reference (non-blocking signals after a charge):**
- `credits.overdraft` — balance went negative (overdraft mode)
- `credits.floor_breach` — balance slipped below `min_balance` without going negative (strict mode under-estimate)
- `credits.low_balance` — edge-triggered threshold crossing
- `credits.cap_warning` — soft spend-cap crossed at settle time

See also: [Financial Safety guide](/docs/financial-safety) · [Notebook 11 — lease lifecycle deep-dive](/docs/notebooks/financial_safety)

In [ ]:
cleanup(pgdata)